# Tune `topk_xgb`

Hub preprocess + XGBoost. Repeated stratified CV; writes [`data/processed/tuned/topk_xgb.json`](../data/processed/tuned/topk_xgb.json).

**Stage 1:** `top_k`, `n_hubs`, `max_depth`, `learning_rate` — max mean PR AUC.

**Stage 2:** threshold — min mean BER.

**Shared preprocess:** median impute → Spearman cluster → RF top-k → Hotelling T² → hub pair interactions (auxiliary features pass through).


In [ ]:
import importlib

import pandas as pd


import secom.tuning.registry as tuning_registry

importlib.reload(tuning_registry)
from secom.pipelines import TARGET_COL, feature_columns, load_mart, split_train_test
from secom.tuning.registry import (
    MODEL_SPECS,
    fit_with_progress,
    run_grid_search,
    save_tuned_params,
    summarize_cv_search,
    tune_classifier_threshold_profiles,
    tuned_params_path,
)

MODEL_ID = "topk_xgb"
spec = MODEL_SPECS[MODEL_ID]


In [ ]:
df = load_mart()
feature_cols = feature_columns(df)
train_df, test_df = split_train_test(df)
X_train = train_df[feature_cols]
y_train = train_df[TARGET_COL].astype(int)
print(len(X_train), "train rows", len(test_df), "test rows (holdout, not used here)")


In [ ]:
param_grid = spec.make_param_grid()
pd.DataFrame([{k: v} for k, v in param_grid.items()])


In [ ]:
search, n_candidates, n_splits, total_fits = run_grid_search(spec, X_train, y_train)
print(f"{MODEL_ID}: {n_candidates} candidates x {n_splits} folds = {total_fits} fits")
search = fit_with_progress(search, X_train, y_train)


In [ ]:
cv_summary, fold_results, aggregated = summarize_cv_search(search, spec)
print("Stage 1 best (mean PR AUC):")
display(aggregated.head(10))


In [ ]:
threshold_result = tune_classifier_threshold_profiles(spec, X_train, y_train, cv_summary)
for pid, prof in threshold_result["profiles"].items():
    line = f"  {pid}: threshold={prof['best_threshold']:.4f}, mean_ber={prof['mean_ber_percent']:.2f}%"
    if "mean_fbeta" in prof:
        line += f", mean_fbeta={prof['mean_fbeta']:.2f}"
    print(line)
display(threshold_result["objective_curves"].head(10))


In [ ]:
payload = save_tuned_params(
    spec,
    cv_summary,
    fold_results,
    aggregated,
    threshold_result=threshold_result,
)
out_path = tuned_params_path(MODEL_ID)
print(f"Wrote {out_path}")
payload["grid_search_best_params"]
